In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Project data directory
DATA_DIR = Path("../data/raw")

customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(
    DATA_DIR / "product_category_name_translation.csv"
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in tables.items():
    print(f"{name:25} Rows: {len(df):>8,} | Columns: {len(df.columns)}")

customers                 Rows:   99,441 | Columns: 5
orders                    Rows:   99,441 | Columns: 8
order_items               Rows:  112,650 | Columns: 7
payments                  Rows:  103,886 | Columns: 5
reviews                   Rows:   99,224 | Columns: 7
products                  Rows:   32,951 | Columns: 9
sellers                   Rows:    3,095 | Columns: 4
geolocation               Rows: 1,000,163 | Columns: 5
category_translation      Rows:       71 | Columns: 2


In [4]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [5]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [6]:
quality_reports = []

for name, df in tables.items():
    report = pd.DataFrame({
        "column": df.columns,
        "data_type": df.dtypes.astype(str).values,
        "missing_count": df.isnull().sum().values,
        "missing_percent": (
            df.isnull().mean().values * 100
        ).round(2),
        "unique_values": df.nunique().values
    })

    report.insert(0, "table", name)
    quality_reports.append(report)

quality_report = pd.concat(
    quality_reports,
    ignore_index=True
)

quality_report.head(20)

,table,column,data_type,missing_count,missing_percent,unique_values
0,customers,customer_id,str,0,0.00,99441
1,customers,customer_unique_id,str,0,0.00,96096
2,customers,customer_zip_code_prefix,int64,0,0.00,14994
3,customers,customer_city,str,0,0.00,4119
4,customers,customer_state,str,0,0.00,27
5,orders,order_id,str,0,0.00,99441
6,orders,customer_id,str,0,0.00,99441
7,orders,order_status,str,0,0.00,8
8,orders,order_purchase_timestamp,str,0,0.00,98875
9,orders,order_approved_at,str,160,0.16,90733


In [7]:
quality_report[
    quality_report["missing_count"] > 0
].sort_values(
    "missing_percent",
    ascending=False
)

,table,column,data_type,missing_count,missing_percent,unique_values
28,reviews,review_comment_title,str,87656,88.34,4527
29,reviews,review_comment_message,str,58247,58.70,36159
11,orders,order_delivered_customer_date,str,2965,2.98,95664
34,products,product_name_lenght,float64,610,1.85,66
33,products,product_category_name,str,610,1.85,73
35,products,product_description_lenght,float64,610,1.85,2960
36,products,product_photos_qty,float64,610,1.85,19
10,orders,order_delivered_carrier_date,str,1783,1.79,81018
9,orders,order_approved_at,str,160,0.16,90733
37,products,product_weight_g,float64,2,0.01,2204


In [8]:

# Identify date columns
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert date columns safely
for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

# Verify data types
print(orders[date_columns].dtypes)

# Check missing dates after conversion
print("\nMissing dates after conversion:")
print(orders[date_columns].isna().sum())


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Missing dates after conversion:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [9]:

# Check product records with missing attributes
product_missing = products[
    products.isna().any(axis=1)
]

print("Products with at least one missing value:",
      len(product_missing))

print("\nMissing values by product column:")
print(products.isna().sum())

print("\nSample of affected products:")
display(product_missing.head(10))


Products with at least one missing value: 611

Missing values by product column:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Sample of affected products:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [10]:

print("Total geolocation rows:", len(geolocation))
print("Exact duplicate rows:", geolocation.duplicated().sum())

print(
    "Unique latitude-longitude pairs:",
    geolocation[
        ["geolocation_lat", "geolocation_lng"]
    ].drop_duplicates().shape[0]
)

print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)


Total geolocation rows: 1000163
Exact duplicate rows: 261831
Unique latitude-longitude pairs: 718463
Unique ZIP prefixes: 19015


In [11]:

# Identify missing product attributes
product_cols = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

# Count missing attributes per product
missing_pattern = (
    products[product_cols]
    .isna()
    .value_counts()
    .rename("product_count")
    .reset_index()
)

print("Missing-value patterns:")
display(missing_pattern)

# Check whether the four attributes are missing together
all_four_missing = products[product_cols].isna().all(axis=1)

print("\nProducts missing all four attributes:",
      all_four_missing.sum())

# Check whether affected products appear in order_items
affected_ids = products.loc[all_four_missing, "product_id"]

used_in_orders = order_items["product_id"].isin(affected_ids)

print("Order-item rows involving these products:",
      used_in_orders.sum())

print("Distinct affected products appearing in order items:",
      order_items.loc[used_in_orders, "product_id"].nunique())


Missing-value patterns:


,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_count
0,False,False,False,False,32341
1,True,True,True,True,610



Products missing all four attributes: 610
Order-item rows involving these products: 1603
Distinct affected products appearing in order items: 610


In [12]:

# Check for impossible or suspicious product dimensions
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print(products[numeric_cols].describe().T)

# Count zero or negative values, excluding missing values
for col in numeric_cols:
    invalid = products[col].notna() & (products[col] <= 0)
    print(f"{col}: {invalid.sum()} zero or negative values")


                              count         mean          std  min    25%  \
product_name_lenght         32341.0    48.476949    10.245741  5.0   42.0   
product_description_lenght  32341.0   771.495285   635.115225  4.0  339.0   
product_photos_qty          32341.0     2.188986     1.736766  1.0    1.0   
product_weight_g            32949.0  2276.472488  4282.038731  0.0  300.0   
product_length_cm           32949.0    30.815078    16.914458  7.0   18.0   
product_height_cm           32949.0    16.937661    13.637554  2.0    8.0   
product_width_cm            32949.0    23.196728    12.079047  6.0   15.0   

                              50%     75%      max  
product_name_lenght          51.0    57.0     76.0  
product_description_lenght  595.0   972.0   3992.0  
product_photos_qty            1.0     3.0     20.0  
product_weight_g            700.0  1900.0  40425.0  
product_length_cm            25.0    38.0    105.0  
product_height_cm            13.0    21.0    105.0  
product_widt

In [13]:

# Find products with zero or negative weight
invalid_weight_products = products[
    products["product_weight_g"].notna()
    & (products["product_weight_g"] <= 0)
]

display(
    invalid_weight_products[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ]
)


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0.0,30.0,25.0,30.0


In [14]:

# Create independent working copies
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
geolocation_clean = geolocation.copy()
category_translation_clean = category_translation.copy()

print("Working copies created successfully.")


Working copies created successfully.


In [15]:

products_clean["product_category_name_analysis"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

print(
    products_clean[
        [
            "product_category_name",
            "product_category_name_analysis"
        ]
    ].head()
)


   product_category_name product_category_name_analysis
0             perfumaria                     perfumaria
1                  artes                          artes
2          esporte_lazer                  esporte_lazer
3                  bebes                          bebes
4  utilidades_domesticas          utilidades_domesticas


In [16]:

from pathlib import Path

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

clean_tables = {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean,
    "geolocation": geolocation_clean,
    "category_translation": category_translation_clean
}

for name, df in clean_tables.items():
    df.to_csv(OUTPUT_DIR / f"{name}_clean.csv", index=False)

print("Working datasets saved to:", OUTPUT_DIR.resolve())


Working datasets saved to: D:\Data_Analyst_Projects\Ecommerce-Customer-Analytics\data\Processed


### Data Quality Finding: Zero Product Weights

Four products in the `cama_mesa_banho` category have a recorded product weight of zero grams.

These values may represent missing or incorrectly recorded product measurements. The affected products are retained because they appear in the order-items dataset.

**Decision:** Preserve the original weight values and flag the affected products for investigation. Do not replace zero weights with arbitrary values.

**Business impact:** These records could affect shipping and product-dimension analysis, but they should not automatically be excluded from sales analysis.

In [17]:

# Flag invalid product weights without deleting records
products_clean["weight_quality_flag"] = np.where(
    products_clean["product_weight_g"].isna(),
    "Missing weight",
    np.where(
        products_clean["product_weight_g"] <= 0,
        "Invalid weight",
        "Valid weight"
    )
)

print(products_clean["weight_quality_flag"].value_counts())

# Inspect flagged products
display(
    products_clean.loc[
        products_clean["weight_quality_flag"] != "Valid weight",
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "weight_quality_flag"
        ]
    ]
)


weight_quality_flag
Valid weight      32945
Invalid weight        4
Missing weight        2
Name: count, dtype: int64


,product_id,product_category_name,product_weight_g,weight_quality_flag
8578,09ff539a621711667c43eba6a3bd8466,bebes,NaN,Missing weight
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0.0,Invalid weight
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0.0,Invalid weight
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0.0,Invalid weight
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,Missing weight
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0.0,Invalid weight


In [18]:
products_clean.to_csv(
    OUTPUT_DIR / "products_clean.csv",
    index=False
)

print("Updated products_clean.csv successfully.")

Updated products_clean.csv successfully.


In [19]:
for name in clean_tables:
    path = OUTPUT_DIR / f"{name}_clean.csv"
    print(f"{name}: {'Saved' if path.exists() else 'Missing'}")

customers: Saved
orders: Saved
order_items: Saved
payments: Saved
reviews: Saved
products: Saved
sellers: Saved
geolocation: Saved
category_translation: Saved


In [20]:

# Validate saved CSV files
for name, original_df in clean_tables.items():
    file_path = OUTPUT_DIR / f"{name}_clean.csv"
    saved_df = pd.read_csv(file_path)

    status = (
        "PASS"
        if len(saved_df) == len(original_df)
        else "CHECK ROW COUNT"
    )

    print(
        f"{name:25} "
        f"Original: {len(original_df):>9,} | "
        f"Saved: {len(saved_df):>9,} | "
        f"{status}"
    )


customers                 Original:    99,441 | Saved:    99,441 | PASS
orders                    Original:    99,441 | Saved:    99,441 | PASS
order_items               Original:   112,650 | Saved:   112,650 | PASS
payments                  Original:   103,886 | Saved:   103,886 | PASS
reviews                   Original:    99,224 | Saved:    99,224 | PASS
products                  Original:    32,951 | Saved:    32,951 | PASS
sellers                   Original:     3,095 | Saved:     3,095 | PASS
geolocation               Original: 1,000,163 | Saved: 1,000,163 | PASS
category_translation      Original:        71 | Saved:        71 | PASS
